# 🚀 Notebook do Professor (Demo) — Aula 10: Context Engineering para agentes

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 10/14 — Módulo 3: curadoria em loop agêntico · MCP overview**  
**⏱️ 1h40min**  
**🧠 Scratchpad · Context rot · MCP**  
**🔁 Andaime 55%**  

---

## 🎯 Objetivo da aula

Entender por que o contexto em agentes é um recurso ainda mais crítico que em chats simples — e dominar as estratégias de curadoria que mantêm a qualidade do agente em loops longos. Preparar o agente para a integração final da Aula 11.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz o lab do aluno com o gabarito das lacunas.

---

# 🔬 Código da aula — slide a slide

### Slide 08 — Scratchpad — o contexto de trabalho do agente

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
# O que o AgentExecutor mantém internamente como scratchpad
scratchpad_turno_1 = """
Thought: Preciso buscar o prazo de garantia nos documentos.
Action: buscar_nos_documentos
Action Input: "prazo de garantia"
Observation: [manual.pdf, pág.15] O prazo de garantia é de 24 meses...
"""  # ~80 tokens

scratchpad_turno_2 = """
Thought: Encontrei 24 meses. Vou converter para dias.
Action: calcular
Action Input: "24 * 30"
Observation: 720
"""  # +40 tokens → total: ~120 tokens

# Após 10 iterações com resultados de busca ricos:
# scratchpad ≈ 10 × 800 chars ≈ ~2.000 tokens só de Observations
# + Thoughts (~500 tokens) + system + tools_schema (~1.000 tokens)
# Total: ~3.500–5.000 tokens por invocação do agente

# Inspecionar o scratchpad via intermediate_steps
resultado = executor.invoke({"input": "pergunta"})
for i, (action, obs) in enumerate(resultado["intermediate_steps"], 1):
    print(f"Iteração {i}: tool={action.tool}, obs_len={len(str(obs))} chars")
# → Iteração 1: tool=buscar_nos_documentos, obs_len=842 chars
# → Iteração 2: tool=calcular, obs_len=3 chars

### Slide 10 — Medir context rot em agente — qualidade vs. turnos

In [ ]:
import tiktoken, matplotlib.pyplot as plt

enc = tiktoken.encoding_for_model("gpt-4")

def contar_tokens_scratchpad(intermediate_steps) -> int:
    """Conta tokens acumulados no scratchpad."""
    texto = ""
    for action, obs in intermediate_steps:
        texto += f"{action.log}\n{obs}\n"
    return len(enc.encode(texto))

# Simular conversação longa — 15 perguntas em sequência
historico_scratchpad = []  # acumula manualmente para simular o contexto crescente
tokens_por_turno      = []
qualidade_por_turno   = []

for i, pergunta in enumerate(PERGUNTAS_TESTE, 1):
    resultado = executor.invoke({"input": pergunta})
    steps    = resultado["intermediate_steps"]
    tokens   = contar_tokens_scratchpad(steps)

    # Avaliar qualidade com LLM-as-judge (Aula 07)
    qualidade = faithfulness_manual(pergunta, resultado["output"],
                                     "\n".join(str(o) for _,o in steps))
    tokens_por_turno.append(tokens)
    qualidade_por_turno.append(qualidade)
    print(f"Turno {i:2d}: {tokens:5d} tokens · qualidade={qualidade:.2f}")

# Plotar context rot
fig, ax1 = plt.subplots(figsize=(9,4))
ax2 = ax1.twinx()
ax1.bar(range(len(tokens_por_turno)), tokens_por_turno, alpha=.4, color="steelblue", label="Tokens")
ax2.plot(qualidade_por_turno, color="#ED145B", marker="o", label="Qualidade")
plt.title("Context rot — tokens vs. qualidade por turno")
plt.show()

### Slide 13 — Compressão de Observations — resumir antes de inserir

In [ ]:
# Tool wrapper que comprime a Observation antes de retornar ao scratchpad
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm_mini = ChatOllama(model="gpt-oss:120b", temperature=0)

chain_resumir = (
    ChatPromptTemplate.from_template(
        "Resuma o texto abaixo em no máximo 3 frases, mantendo APENAS os fatos essenciais:\n\n{texto}"
    )
    | llm_mini | StrOutputParser()
)

@tool
def buscar_na_web_comprimida(query: str) -> str:
    """Use para buscar informações atuais na web. Retorna um resumo conciso dos resultados."""
    resultado_bruto = DuckDuckGoSearchRun().run(query)

    # Comprime ANTES de retornar ao scratchpad
    if len(resultado_bruto) > 500:
        return chain_resumir.invoke({"texto": resultado_bruto})
    return resultado_bruto  # já curto — não precisa resumir

# Diferença típica:
# Sem compressão: Observation = 800–1200 chars (~250 tokens)
# Com compressão: Observation = 150–250 chars (~55 tokens)
# Redução: ~78% dos tokens de Observation

### Slide 14 — Janela deslizante — custo fixo em loops longos

In [ ]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.messages import trim_messages

# Estratégia 1 — limitar via max_iterations (simples)
executor_limitado = AgentExecutor(
    agent=agente,
    tools=tools,
    max_iterations=5,       # mantém no máximo 5 iterações no scratchpad
    max_execution_time=60,  # para após 60 segundos (fallback de segurança)
    verbose=True,
)

# Estratégia 2 — resumo episódico entre turnos (avançado)
def resumir_scratchpad(intermediate_steps: list, max_steps: int = 3) -> list:
    """Mantém as últimas max_steps iterações e resume o resto."""
    if len(intermediate_steps) <= max_steps:
        return intermediate_steps

    antigas   = intermediate_steps[:-max_steps]
    recentes  = intermediate_steps[-max_steps:]

    # Resumir as iterações antigas em uma linha
    resumo = ", ".join(f"{a.tool}→{str(o)[:50]}" for a,o in antigas)
    print(f"[Resumo de {len(antigas)} iterações anteriores: {resumo}]")

    # Retorna só as recentes — scratchpad compacto
    return recentes

### Slide 15 — Memória episódica — persistir entre sessões

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from datetime import datetime

# Salvar resumo da sessão ao encerrar
def salvar_sessao(pergunta: str, resposta: str, session_id: str):
    """Persiste o par (pergunta, resposta) como memória episódica."""
    db_episodico.add_documents([Document(
        page_content=f"Pergunta: {pergunta}\nResposta: {resposta}",
        metadata={
            "session_id": session_id,
            "timestamp": datetime.now().isoformat(),
            "tipo": "episodio",
        },
    )])

# Recuperar contexto de sessões anteriores (memória episódica como tool)
@tool
def lembrar_sessoes_anteriores(query: str) -> str:
    """Use quando o usuário se referir a algo discutido em conversa anterior.
    Retorna resumos de interações passadas relevantes para a query."""
    docs = db_episodico.similarity_search(query, k=2,
                                          filter={"tipo": "episodio"})
    return "\n\n".join(d.page_content for d in docs)

### Slide 18 — MCP na prática — o seu agente como cliente MCP

In [ ]:
!pip install langchain-mcp-adapters -q  # adaptador oficial LangChain ↔ MCP

from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession
from mcp.client.stdio import stdio_client

# Conectar a um servidor MCP local (ex: servidor de filesystem)
async def conectar_mcp():
    async with stdio_client(
        {"command": "npx", "args": ["@modelcontextprotocol/server-filesystem", "/content"]}
    ) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            # Carrega tools do servidor MCP como tools LangChain
            mcp_tools = await load_mcp_tools(session)
            # → lista de ferramentas prontas para o AgentExecutor
            return mcp_tools

# Servidores MCP gratuitos disponíveis hoje:
# @modelcontextprotocol/server-filesystem  — ler/escrever arquivos
# @modelcontextprotocol/server-brave-search — busca web
# mcp-server-fetch                          — fazer requisições HTTP
# mcp-server-sqlite                         — consultar SQLite
# Catálogo: github.com/modelcontextprotocol/servers

### Slide 22 — Python novo desta aula

In [ ]:
# 1. ax.twinx() — segundo eixo Y no mesmo gráfico matplotlib
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()          # eixo direito — mesmo x, y diferente
ax1.bar(x, tokens, color="steelblue", alpha=.5)  # barras no eixo esquerdo
ax2.plot(qualidade, color="#ED145B", marker="o")  # linha no eixo direito

# 2. list slice negativo — últimas N iterações
steps   = ["a","b","c","d","e"]
recentes = steps[-3:]   # → ["c", "d", "e"] (últimas 3)
antigas  = steps[:-3]   # → ["a", "b"] (exceto as últimas 3)

# 3. datetime.now().isoformat() — timestamp para metadados
from datetime import datetime
ts = datetime.now().isoformat()  # → "2026-07-13T14:35:22.123456"

# 4. plt.bar com offset para barras lado a lado
x     = range(5)       # posições 0, 1, 2, 3, 4
width = 0.4
plt.bar([i-width/2 for i in x], vals_a, width, label="A")  # deslocado -0.2
plt.bar([i+width/2 for i in x], vals_b, width, label="B")  # deslocado +0.2

# 5. statistics.mean() para calcular redução média
import statistics
reducao_pct = (1 - statistics.mean(tok_com) / statistics.mean(tok_sem)) * 100
print(f"Redução média: {reducao_pct:.1f}%")

---

# 💻 Lab do aluno — versão com lacunas

## 📋 Roteiro do Lab

**Lab — Aula 10 · 2º Semestre**  
### Compressão do scratchpad e medição de impacto ★★★

*Grupo 3–4 · 20 minutos · Google Colab*

1. Complete as 4 lacunas — contar_tokens_scratchpad (concatenar action.log + obs), buscar_na_web_comprimida (invocar chain_resumir com a chave correta), 5 perguntas do domínio e os dois argumentos do plt.bar.
2. Analise o gráfico — em qual pergunta a diferença de tokens foi maior? Por quê? (Dica: resultado de busca web é maior que resultado do RAG)
3. Calcule a redução média em % de tokens usando a fórmula (1 - mean(tok_com)/mean(tok_sem)) * 100. Documente em célula markdown.

> **🎯 Gabarito das lacunas**
>
> Lacuna 1: texto += f"{action.log}\n{obs}\n" ; enc.encode(texto)
>
> Lacuna 2: chain_resumir.invoke({"texto": bruto})
>
> Lacuna 3: 5 perguntas reais do domínio do grupo
>
> Lacuna 4: tok_sem (1º bar) ; tok_com (2º bar)

In [ ]:
!pip install langchain langchain-ollama langchain-community duckduckgo-search tiktoken matplotlib -q

import tiktoken, matplotlib.pyplot as plt
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

enc = tiktoken.encoding_for_model("gpt-4")

# 👉 LACUNA 1: implemente contar_tokens_scratchpad
def contar_tokens_scratchpad(steps) -> int:
    """Soma os tokens de todos os Thoughts + Observations."""
    texto = ""
    for action, obs in steps:
        texto += ___  # concatene action.log e obs
    return len(enc.encode(___))

# 👉 LACUNA 2: implemente a tool de busca com compressão de Observation
@tool
def buscar_na_web_comprimida(query: str) -> str:
    """Use para buscar informações atuais. Retorna resumo conciso."""
    bruto = DuckDuckGoSearchRun().run(query)
    if len(bruto) > 500:
        return chain_resumir.invoke({___: bruto})  # invocar chain de resumo
    return bruto

# 👉 LACUNA 3: rodar as duas versões (sem e com compressão) com 5 perguntas cada
PERGUNTAS = [___, ___, ___, ___, ___]  # 5 perguntas reais do domínio

tok_sem, tok_com = [], []
for q in PERGUNTAS:
    r_sem = executor_sem.invoke({"input":q})  # sem compressão (Aula 09)
    r_com = executor_com.invoke({"input":q})  # com compressão (esta aula)
    tok_sem.append(contar_tokens_scratchpad(r_sem["intermediate_steps"]))
    tok_com.append(contar_tokens_scratchpad(___))

# 👉 LACUNA 4: plotar comparação tokens sem vs. com compressão
x = range(len(PERGUNTAS))
plt.bar([i-.2 for i in x], ___, .4, label="Sem compressão", color="salmon")
plt.bar([i+.2 for i in x], ___, .4, label="Com compressão",  color="steelblue")
plt.legend(); plt.xlabel("Pergunta"); plt.ylabel("Tokens no scratchpad")
plt.title("Impacto da compressão de Observations no scratchpad"); plt.show()

## 📚 Referências da aula

- Blog Anthropic Engineering — "Context Engineering for AI Agents" (setembro 2025). Fonte primária desta aula — princípio da ação mínima, tipos de memória, curadoria de scratchpad. anthropic.com/engineering/context-engineering
- Docs Model Context Protocol — Especificação oficial, servidores disponíveis e guia de implementação. modelcontextprotocol.io
- Paper Liu, N. et al. — "Lost in the Middle: How Language Models Use Long Contexts." EMNLP, 2023. A base empírica do context rot em contextos longos. arxiv.org/abs/2307.03172
- Docs LangChain MCP Adapters — Integrar servidores MCP como tools LangChain. github.com/langchain-ai/langchain-mcp-adapters
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 2 — Agentes e ambientes: a analogia memória=RAM / conhecimento=HD que fundamenta os 3 tipos de memória agêntica.
- Ebook Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 8: Memory Management — a distinção Short-Term vs. Long-Term por trás do scratchpad e da memória episódica desta aula.

---

**Próxima Aula — Aula 11 · 26/10** — Aula Integradora — Agente com RAG + Gradio ao vivo
  
100% lab. Integrar tudo. Publicar URL pública. Entregar CKP03.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*